# CofC Match Review and Publication

This is the complete staff workflow. Change the match slug and reviewer below, choose **Runtime → Run all**, review the displayed validation and scores, then type one final publication confirmation. The notebook automatically prepares staff events, loads missing evidence, archives the source and Match Flow, publishes the reviewed COUG scores, and verifies the result.

## 1. Connect Google Drive

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Local Jupyter session detected; Drive mount skipped.')

## 2. Match setup

For each match, change only these first three values.

In [ ]:
SEASON = '2026'
MATCH_SLUG = '2026-08-23_fgcu'
REVIEWED_BY = 'Anissa Williams'

DRIVE_PIPELINE_ROOT = Path('/content/drive/.shortcut-targets-by-id/1CX5Tm9R4U5YA8vJCOOJ4FgMfKIqeQV1D/CofC_Soccer/data_ingestion_pipeline/2026_pipeline')
MATCH_DIR = DRIVE_PIPELINE_ROOT / SEASON / 'matches' / MATCH_SLUG
SOURCE_DIR = MATCH_DIR / '00_source'
STAFF_DIR = MATCH_DIR / 'staff'
BUNDLE_DIR = MATCH_DIR / '20_generated'
print('Publishing from:', MATCH_DIR)

## 3. Load current tested code and private credentials

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/anissawilliams/cofc-soccer-analytics.git'
if IN_COLAB:
    REPO_ROOT = Path('/content/cofc-soccer-analytics')
    if REPO_ROOT.exists():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'switch', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(REPO_ROOT)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements-backend.txt'), 'rapidfuzz>=3.0.0', 'pypdf>=5.0.0'], check=True)
    from google.colab import userdata
    missing_secrets = []
    for secret_name in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY'):
        try:
            secret_value = userdata.get(secret_name)
        except Exception:
            secret_value = None
        if secret_value:
            os.environ[secret_name] = secret_value
        else:
            missing_secrets.append(secret_name)
    if missing_secrets:
        raise RuntimeError('Add these Colab Secrets and grant notebook access: ' + ', '.join(missing_secrets))
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((path for path in candidates if (path / 'pipeline').is_dir()), None)
    if REPO_ROOT is None:
        raise RuntimeError('Open this notebook from the cofc-soccer-analytics repository.')
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')

revision = subprocess.run(['git', '-C', str(REPO_ROOT), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('Code revision:', revision)
print('Supabase credentials: configured')

## 4. Run the complete workflow

Choose **Runtime → Run all**. The notebook stops automatically on any failed check. At the final prompt, review the score table above it and type `PUBLISH <match slug>` once.

In [ ]:
import hashlib
import json
from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import Markdown, display

def run_step(label, command):
    print(f'\n=== {label} ===')
    result = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'{label} failed. Nothing after this step was run.')
    print('PASS')
    return result

if not REVIEWED_BY.strip():
    raise ValueError('REVIEWED_BY is required.')
PREPARE_INTAKE = REPO_ROOT / 'pipeline' / 'ingestion' / 'prepare_match_intake.py'
ROSTER = REPO_ROOT / 'pipeline' / 'ingestion' / f'roster_{SEASON}.csv'
REPORT_PATH = BUNDLE_DIR / f'{MATCH_SLUG}_intake_report.json'
VALIDATION_PATH = BUNDLE_DIR / f'{MATCH_SLUG}_validation_report.md'
APPROVAL_PATH = BUNDLE_DIR / f'{MATCH_SLUG}_approval.json'
if not SOURCE_DIR.is_dir():
    raise FileNotFoundError(f'Missing source folder: {SOURCE_DIR}')
if not ROSTER.is_file():
    raise FileNotFoundError(f'Missing parser roster: {ROSTER}')
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
prepare_command = [
    sys.executable, str(PREPARE_INTAKE), '--input-dir', str(SOURCE_DIR),
    '--output-dir', str(BUNDLE_DIR), '--season', SEASON, '--slug', MATCH_SLUG,
    '--roster', str(ROSTER),
]
metadata_path = BUNDLE_DIR / f'{MATCH_SLUG}_metadata.json'
if metadata_path.is_file():
    prepare_command.extend(['--metadata', str(metadata_path)])
run_step('Inspect source files and refresh intake readiness', prepare_command)

report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
if report.get('slug') != MATCH_SLUG or str(report.get('season')) != SEASON:
    raise ValueError('Bundle slug/season does not match the setup cell.')
readiness = {
    'match analytics': bool((report.get('analytics') or {}).get('ready')),
    'COUG scoring': bool((report.get('scoring') or {}).get('ready')),
    'official minutes': bool((report.get('minutes') or {}).get('ready')),
}
display(Markdown(VALIDATION_PATH.read_text(encoding='utf-8')))
print('Readiness:', json.dumps(readiness, indent=2))
if (report.get('validation') or {}).get('status') == 'blocked' or not all(readiness.values()):
    raise RuntimeError('Publication stopped after intake review because required products are not ready: ' + json.dumps(readiness))

PREPARE_STAFF = REPO_ROOT / 'pipeline' / 'ingestion' / 'prepare_staff_events.py'
LOAD_STAFF = REPO_ROOT / 'pipeline' / 'ingestion' / 'load_staff_events.py'
PROMOTE = REPO_ROOT / 'pipeline' / 'ingestion' / 'promote_match_intake.py'
LOAD = REPO_ROOT / 'pipeline' / 'ingestion' / 'load_match.py'
PUBLISH = REPO_ROOT / 'pipeline' / 'analytics' / 'publish_event_derived_coug_scores.py'
SCORE_REVIEW_DIR = BUNDLE_DIR / 'score_review'
STAFF_CSV = STAFF_DIR / 'staff_events.csv'

if STAFF_CSV.is_file():
    run_step('Validate staff events', [
        sys.executable, str(PREPARE_STAFF), '--season', SEASON, '--slug', MATCH_SLUG,
        '--staff-dir', str(STAFF_DIR), '--output-dir', str(BUNDLE_DIR),
    ])
else:
    print('\nNo staff event CSV supplied; continuing without manual incidents.')

approval = {
    'schema_version': 1,
    'match_slug': MATCH_SLUG,
    'season': SEASON,
    'intake_report_sha256': hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest(),
    'reviewed_by': REVIEWED_BY.strip(),
    'reviewed_at': datetime.now(ZoneInfo('America/New_York')).isoformat(),
    'approvals': {'source_archive': True, 'match_analytics': True, 'coug_scoring': True},
    'notes': 'Approved through the single-run staff publication notebook.',
}
APPROVAL_PATH.write_text(json.dumps(approval, indent=2, sort_keys=True), encoding='utf-8')

run_step('Archive preflight', [
    sys.executable, str(PROMOTE), '--source-dir', str(SOURCE_DIR), '--bundle-dir', str(BUNDLE_DIR),
])
run_step('Database evidence preflight', [
    sys.executable, str(LOAD), '--slug', MATCH_SLUG, '--season', SEASON,
    '--bundle-dir', str(BUNDLE_DIR), '--dry-run',
])
run_step('Load database evidence', [
    sys.executable, str(LOAD), '--slug', MATCH_SLUG, '--season', SEASON,
    '--bundle-dir', str(BUNDLE_DIR),
])

if STAFF_CSV.is_file():
    run_step('Staff event preview', [
        sys.executable, str(LOAD_STAFF), '--season', SEASON, '--slug', MATCH_SLUG,
        '--staff-dir', str(STAFF_DIR),
    ])
    run_step('Apply staff events', [
        sys.executable, str(LOAD_STAFF), '--season', SEASON, '--slug', MATCH_SLUG,
        '--staff-dir', str(STAFF_DIR), '--apply',
    ])

run_step('Archive sources and publish Match Flow', [
    sys.executable, str(PROMOTE), '--source-dir', str(SOURCE_DIR),
    '--bundle-dir', str(BUNDLE_DIR), '--apply',
])
run_step('Final COUG score preview', [
    sys.executable, str(PUBLISH), '--season', SEASON, '--slug', MATCH_SLUG,
    '--output-root', str(SCORE_REVIEW_DIR),
])

expected = f'PUBLISH {MATCH_SLUG}'
confirmation = input(f'\nReview the scores above. Type {expected!r} to publish, or press Enter to stop safely: ').strip()
if confirmation != expected:
    raise RuntimeError('COUG score publication cancelled. Evidence and archive are safe; public scores were not changed.')

run_step('Fresh score safety check', [
    sys.executable, str(PUBLISH), '--season', SEASON, '--slug', MATCH_SLUG,
    '--output-root', str(SCORE_REVIEW_DIR),
])
run_step('Publish COUG scores', [
    sys.executable, str(PUBLISH), '--season', SEASON, '--slug', MATCH_SLUG,
    '--output-root', str(SCORE_REVIEW_DIR), '--apply',
])

from supabase import create_client
client = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_KEY'])
sessions = client.table('session').select('id,notes').eq('session_date', MATCH_SLUG[:10]).eq('season', SEASON).execute().data or []
sessions = [row for row in sessions if f'slug: {MATCH_SLUG}' in str(row.get('notes') or '').splitlines()]
if len(sessions) != 1:
    raise RuntimeError(f'Final verification found {len(sessions)} sessions; expected one.')
session_id = sessions[0]['id']
matches = client.table('match').select('id').eq('session_id', session_id).execute().data or []
scores = client.table('coug_score').select('id').eq('session_id', session_id).eq('score_type', 'match').execute().data or []
artifacts = client.table('source_file').select('id,source_type').eq('session_id', session_id).eq('is_active', True).execute().data or []
flow_count = sum(row.get('source_type') == 'match_flow' for row in artifacts)
if len(matches) != 1 or not scores or flow_count != 1:
    raise RuntimeError(f'Final verification failed: matches={len(matches)}, scores={len(scores)}, match_flow={flow_count}.')
print('\n✅ PUBLISHED AND VERIFIED')
print(f'Match: {MATCH_SLUG}')
print(f'COUG player scores: {len(scores)}')
print(f'Archived artifacts: {len(artifacts)}')
print('Match Flow: ready')

## Finished

A successful run ends with **PUBLISHED AND VERIFIED**. If any earlier step fails, stop and share that single failed section; completed writes are idempotent and the public COUG scores remain unchanged until the final confirmation.